# Initializing packages 

In [25]:
import os
import json
import time
from dotenv import load_dotenv
from typing import Annotated
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict
from langchain_mcp_adapters.client import MultiServerMCPClient

from agentevals.trajectory.match import create_trajectory_match_evaluator

load_dotenv()  # loads BLABLADOR_API_KEY from .env

# DeepEval tracing imports
# from deepeval.tracing import observe, update_current_span, update_current_trace  #for production monitoring
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
#from deepeval.tracing import get_trace_stack
from deepeval import evaluate
from deepeval.dataset import Golden
from deepeval.models import LiteLLMModel


from deepeval.metrics import (
    # RAG / Retrieval metrics
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    ContextualRelevancyMetric,

    # Agent / Tool metrics
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    TaskCompletionMetric,
    step_efficiency,

    # General quality
    HallucinationMetric,
    MisuseMetric,
)
from deepeval.evaluate.configs import AsyncConfig

In [26]:
BLABLADOR_BASE_URL = "https://api.helmholtz-blablador.fz-juelich.de/v1"
blablador_key=os.getenv("BLABLADOR_API_KEY")
Chat_AI_Base_URL = "https://chat-ai.academiccloud.de/v1"
chat_ai_key=os.getenv("Chat_AI_API_KEY")


In [27]:
import requests

headers = {
    "Authorization": f"Bearer {chat_ai_key}",
    "Content-Type": "application/json"
}

response = requests.post(Chat_AI_Base_URL, headers=headers)

print("Limit/min:", response.headers.get("x-ratelimit-limit-minute"))
print("Remaining/min:", response.headers.get("x-ratelimit-remaining-minute"))
print("Reset (s):", response.headers.get("ratelimit-reset"))

Limit/min: 30
Remaining/min: 29
Reset (s): 14


In [ ]:
from blablador import Models 

models = Models(blablador_key).get_model_ids()
models

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=chat_ai_key,
    base_url=Chat_AI_Base_URL
)

models = client.models.list()
for m in models.data:
    print(m.id)


# Agent with Tools via MCP server Initilization

In [29]:
#React-agent
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    
with open("mcp_config.json") as f:
    config = json.load(f)

    client = MultiServerMCPClient(config)
    tools = await client.get_tools()
    print(f"Loaded {len(tools)} tools: {[t.name for t in tools]}")

    llm = ChatOpenAI(
        model="01 - MiniMax-M2.7 - our best model as of April, 2026",
        base_url=BLABLADOR_BASE_URL,
        api_key=blablador_key,
    )
    llm_with_tools = llm.bind_tools(tools)
    print("Model ready.")

    async def agent_node(state: AgentState) -> AgentState:
        response = await llm_with_tools.ainvoke(state["messages"])  # must await
        return {"messages": [response]}

    def build_graph():
        graph = StateGraph(AgentState)
        graph.add_node("agent", agent_node)
        graph.add_node("tools", ToolNode(tools))
        graph.add_edge(START, "agent")
        graph.add_conditional_edges("agent", tools_condition)
        graph.add_edge("tools", "agent")
        return graph.compile()

    app = build_graph()
    print("Graph compiled successfully.")


Loaded 2 tools: ['search_UFZ_guidelines', 'search_funding_guidelines']
Model ready.
Graph compiled successfully.


# Agent execution function that extracts actual output, retrieval context, and tools called for evaluation


In [30]:

async def run_agent(user_input: str) -> dict:
    result = await app.ainvoke({"messages": [HumanMessage(content=user_input)]})
    messages = result["messages"]
    
    # 1. actual_output — last AIMessage that has content (not a tool call)
    actual_output = next(
        msg.content
        for msg in reversed(messages)
        if isinstance(msg, AIMessage) and msg.content
    )

    # 2. retrieval_context — what your tools returned
    retrieval_context = [
        str(msg.content) for msg in messages
        if isinstance(msg, ToolMessage)
    ]

    # 3. tools_called — pair each AIMessage tool call with its ToolMessage output
    tools_called = []
    tool_outputs = [msg for msg in messages if isinstance(msg, ToolMessage)]
    tool_idx = 0

    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                output = tool_outputs[tool_idx].content if tool_idx < len(tool_outputs) else ""
                tools_called.append(ToolCall(
                    name=tc["name"],
                    input_parameters=tc.get("args") or {},   # <-- renamed + defensive fallback
                    output=str(output),
                ))
                tool_idx += 1

    return {
        "actual_output": actual_output,
        "retrieval_context": retrieval_context,
        "tools_called": tools_called,
        "messages": messages,
    }

In [ ]:
# queries = ["What are the UFZ guidelines for long-term archiving?",]
# if False:  # set to True to run the agent on the queries
#     for q in queries:
#         print(f"Q: {q}")
#         print(f"A: {await run_agent(q)}")   # must await async run()
#         print()

# Judge for evaluation metrics
#### MoE variants (same architecture):
#### Qwen/Qwen3.5-35B-A3B — 35B total, 3B active
#### Qwen/Qwen3.5-122B-A10B — 122B / 10B active
#### Qwen/Qwen3.5-397B-A17B — 397B / 17B active

In [31]:

judge = LiteLLMModel(
    #model= "openai/alias-fast",          # prefix with "openai/" for OpenAI-compat APIs 
    model= "openai/qwen3.5-122b-a10b",     #02 - Qwen3.5-122B-A10B-FP8, general purpose large model
    base_url=Chat_AI_Base_URL,
    api_key=chat_ai_key
)


# Metrics configrations
#### A metric is only successful if the evaluation score is equal to or greater than threshold, which is defaulted to 0.5 for all metrics.

In [32]:
enableverbose = True

tool_correctness = ToolCorrectnessMetric( threshold=0.5, model=judge, include_reason=True,verbose_mode=enableverbose)
argument_correctness = ArgumentCorrectnessMetric(threshold=0.5, model=judge, include_reason=True,verbose_mode=enableverbose)
task_completion = TaskCompletionMetric(threshold=0.5, model=judge, include_reason=True,verbose_mode=enableverbose)
faithfulness    = FaithfulnessMetric(threshold=0.7, model=judge, include_reason=True,verbose_mode=enableverbose)
answer_rel      = AnswerRelevancyMetric(threshold=0.7, model=judge, include_reason=True,verbose_mode=enableverbose)
hallucination   = HallucinationMetric(threshold=0.5, model=judge, include_reason=True,verbose_mode=enableverbose)
contextual_recall = ContextualRecallMetric(threshold=0.7, model=judge, include_reason=True,verbose_mode=enableverbose)
contextual_precision = ContextualPrecisionMetric(threshold=0.7, model=judge, include_reason=True,verbose_mode=enableverbose)
contextual_relevancy = ContextualRelevancyMetric(threshold=0.7, model=judge, include_reason=True,verbose_mode=enableverbose)
misuse = MisuseMetric(threshold=0.5, model=judge,domain="research data management")

In [33]:
def dict_to_toolcall(d: dict) -> ToolCall:
    return ToolCall(
        name=d["name"],
        input_parameters=d.get("input_parameters") or {},
        output=d.get("output"),
    )

def dict_to_testcase(d: dict) -> LLMTestCase:
    return LLMTestCase(
        input=d["input"],
        actual_output=d["actual_output"],
        expected_output=d.get("expected_output"),
        context=d.get("context"),
        retrieval_context=d.get("retrieval_context"),
        tools_called=[dict_to_toolcall(t) for t in d.get("tools_called", [])],
        expected_tools=[dict_to_toolcall(t) for t in d.get("expected_tools", [])],
    )

with open("test_cases_built3.json", encoding="utf-8") as f:
    raw = json.load(f)

loaded_cases = [dict_to_testcase(d) for d in raw]
meta = [{"id": d["id"], "type": d["type"]} for d in raw]

print(f"Loaded {len(loaded_cases)} test cases")

Loaded 12 test cases


In [34]:
case= loaded_cases[0]  # pick one for now
case

LLMTestCase(input='Here is the storage and backup section from a project\'s DMP:\n\n"Data will be stored in the mTox group folder and image data will be routinely uploaded in OMERO after processing. On yearly bases, data which are not anymore necessary will be transferred to 1) the UFZ central archiving systems or 2) the dCache folder of the Helmholtz (InfiniteSpace) for backup and long-term storage."\n\nDoes this satisfy UFZ storage and archiving requirements?', actual_output='<think>Based on the information I\'ve gathered, I now have a good understanding of what the UFZ guidelines say about storage and archiving. Let me summarize what I found:\n\n1. **Data Lifecycle**: The UFZ follows a data lifecycle model that includes stages: data generation, data preparation, data evaluation/analysis, storage and long-term archiving, and provision of data through publication.\n\n2. **Backup**: The guidelines emphasize that:\n   - Backup means creating a copy of data on another storage device\n   

In [35]:
THROTTLE_CONFIG = {
  "base_duration": 60,
  "max_duration": 240,
  "multiplier": 2
}

from litellm.exceptions import RateLimitError, TimeoutError

attempt = 0
max_retries = 3

# while True:
try:
    results = evaluate(
        test_cases=[case],
        metrics=[
            tool_correctness,
            argument_correctness,
            task_completion,
            faithfulness,
            answer_rel,
            hallucination,
            contextual_recall,
            contextual_precision,
            contextual_relevancy,
            misuse
        ],
        # error_config={"DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE": 60.0 },
    )
    print(f"completed evaluation for test case {results[0].test_case.id}")
except RateLimitError as rle:
    print(f"Rate limit exceeded: {str(rle)}")
    print(rle.message)
    print(**rle)
    print(rle.headers)
except TimeoutError as te:
    print(f"Timeout error: {str(te)}")
    print(te.message)
except Exception as e:
    print(f"Error during evaluation: {type(e).__name__}, {str(e)}")
    if attempt == max_retries:
        print(f"Max retries reached for test case")
    else:
        delay = min(THROTTLE_CONFIG["base_duration"] * (THROTTLE_CONFIG["multiplier"] ** attempt),
                    THROTTLE_CONFIG["max_duration"])
        print(f"Rate limited, waiting {delay}s (attempt {attempt+1})")
        time.sleep(delay)
        attempt += 1


ImportError: cannot import name 'TimeoutError' from 'litellm.exceptions' (c:\Users\abdul\langgraph-agent\.venv\Lib\site-packages\litellm\exceptions.py)

In [ ]:
THROTTLE_CONFIG = {
  "base_duration": 60,
  "max_duration": 240,
  "multiplier": 2
}

def evaluate_with_backoff(tc, config):
    attempt = 0
    max_retries = 3
    while True:
        try:
            for tc in enumerate(loaded_cases):
                results = evaluate(test_cases=[tc], metrics=[answer_rel,faithfulness],
                                async_config=AsyncConfig(run_async=False))
                print(f"completed evaluation for test case {results[0].test_case.id}")
        except Exception as e:
            print(f"Error during evaluation: {type(e).__name__}, {str(e)}")
            if attempt == max_retries:
                print(f"Max retries reached for test case")
            else:
                delay = min(config["base_duration"] * (config["multiplier"] ** attempt),
                            config["max_duration"])
                print(f"Rate limited, waiting {delay}s (attempt {attempt+1})")
                time.sleep(delay)
                attempt += 1


In [ ]:
results = evaluate_with_backoff(loaded_cases, config=THROTTLE_CONFIG)

In [ ]:
# # when change something in test_cases.py, reload it to reflect changes in the notebook
# import importlib, middle_test_cases
# importlib.reload(middle_test_cases)

In [ ]:
# from middle_test_cases import TEST_CASES, build_test_case

In [ ]:
# def build_query(tc: dict) -> str:
#     if "{source_extract}" in tc["input_template"]:
#         return tc["input_template"].format(source_extract=tc.get("source_extract", ""))
#     return tc["input_template"]

# test_cases = []
# i = 0
# for i, tc in enumerate(TEST_CASES):
#     print(i + 1)
#     query = build_query(tc)
#     result = await run_agent(query)
#     if result["actual_output"].startswith("[NO FINAL RESPONSE"):
#         print(f"⚠ {tc['id']}: agent produced no final AIMessage content")
#     test_case = build_test_case(tc, result)
#     test_cases.append(test_case)

### Save built test cases into json

In [ ]:
# def toolcall_to_dict(tc: ToolCall) -> dict:
#     return {
#         "name": tc.name,
#         "input_parameters": tc.input_parameters or {},
#         "output": tc.output,
#     }

# def testcase_to_dict(tc: LLMTestCase, tc_id: str = None, tc_type: str = None) -> dict:
#     return {
#         "id": tc_id,
#         "type": tc_type,
#         "input": tc.input,
#         "actual_output": tc.actual_output,
#         "expected_output": tc.expected_output,
#         "context": tc.context,
#         "retrieval_context": tc.retrieval_context,
#         "tools_called": [toolcall_to_dict(t) for t in (tc.tools_called or [])],
#         "expected_tools": [toolcall_to_dict(t) for t in (tc.expected_tools or [])],
#     }

# with open("test_cases_built3.json", "w", encoding="utf-8") as f:
#     json.dump(
#         [testcase_to_dict(tc, tc_id=t["id"], tc_type=t["type"])
#          for tc, t in zip(test_cases, TEST_CASES)],
#         f, indent=2, ensure_ascii=False
#     )

# print(f"Saved {len(test_cases)} built test cases to test_cases_built3.json")

# Load test cases

In [ ]:
def dict_to_toolcall(d: dict) -> ToolCall:
    return ToolCall(
        name=d["name"],
        input_parameters=d.get("input_parameters") or {},
        output=d.get("output"),
    )

def dict_to_testcase(d: dict) -> LLMTestCase:
    return LLMTestCase(
        input=d["input"],
        actual_output=d["actual_output"],
        expected_output=d.get("expected_output"),
        context=d.get("context"),
        retrieval_context=d.get("retrieval_context"),
        tools_called=[dict_to_toolcall(t) for t in d.get("tools_called", [])],
        expected_tools=[dict_to_toolcall(t) for t in d.get("expected_tools", [])],
    )

with open("test_cases_built3.json", encoding="utf-8") as f:
    raw = json.load(f)

loaded_cases = [dict_to_testcase(d) for d in raw]
meta = [{"id": d["id"], "type": d["type"]} for d in raw]

print(f"Loaded {len(loaded_cases)} test cases")

In [ ]:
meta

In [ ]:
THROTTLE_CONFIG = {
  "base_duration": 60,
  "max_duration": 240,
  "multiplier": 2
}

In [ ]:
def get_delay(attempt: int) -> float:
    delay = THROTTLE_CONFIG["base_duration"] * (THROTTLE_CONFIG["multiplier"] ** attempt)
    return min(delay, THROTTLE_CONFIG["max_duration"])

In [ ]:
all_results = []
failed_cases = []   # <-- standalone list of failed case ids
MAX_RETRIES = 3


for i, tc in enumerate(test_case):
    tc_id = meta[i]["id"]
    #print(f"Evaluating test case {i + 1}/{len(test_case)}: {tc_id}")
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            results = evaluate(
                test_cases=[tc],
                metrics=[answer_rel, faithfulness],
                async_config=AsyncConfig(run_async=False)
            )
            all_results.append((results))

        except Exception as e:
            print(f"✗ {tc_id} attempt {attempt} failed: {e}")
            if attempt == MAX_RETRIES:
                failed_cases.append(tc_id)
            else:
                get_delay(attempt)

In [ ]:
all_results_3 = []

for tc in loaded_cases:
    for m in metrics:
      result = m.measure(tc)  # compute metric for this test case
      tc_id = tc["id"]
      name = result.metric_name
      score = result.score
      reason = result.reason
    all_results_3.append({
        "test_case_id": tc_id,
        "metric_name": name,
        "score": score,
        "reason": reason
    })

In [ ]:
print(type(loaded_cases[0]))

In [ ]:
tc = loaded_cases[0]
print(tc.name)                 # str or None
print(tc.additional_metadata)  # dict or None
print(tc.__dict__.keys())      # catch-all — shows every attribute actually set
print(repr(loaded_cases[0].name))
print(loaded_cases[0].__dict__.keys())

In [ ]:
import time

all_results = []
failed_cases = []   # <-- standalone list of failed case ids
MAX_RETRIES = 3

def case_id(tc):
    return tc.metadata["id"], tc.metadata["type"]  # confirm this matches your loader

for tc in enumerate(loaded_cases):
    # tc_id,tc_type = case_id(tc)
    # print(tc_id, tc_type)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            results = evaluate(
                test_cases=[tc],
                metrics=metrics,
                async_config=AsyncConfig(run_async=False)
            )
            all_results.append((results))

            with open("all_results_5.json", "w") as f:
                json.dump([r.model_dump() if hasattr(r, "model_dump") else r for r in all_results], f, indent=2)

            print(f"✓ {tc_id} done")
            break

        except Exception as e:
            print(f"✗ {tc_id} attempt {attempt} failed: {e}")
            if attempt == MAX_RETRIES:
                failed_cases.append(tc_id)
            else:
                time.sleep(60 ** attempt)

print(f"\nDone. {len(failed_cases)} failed case(s): {failed_cases}")

In [ ]:
OUT = "all_results_medium.json"
all_results_3 = json.load(open(OUT)) if os.path.exists(OUT) else []
done = {(r["test_case_id"], r["metric_name"]) for r in all_results_3}

def case_id(tc):
    return tc.metadata["id"], tc.metadata["type"]  # confirm this matches your loader

for m in metrics:                          # metric-outer: fail fast on broken metrics
    name = m.__class__.__name__
    for tc in loaded_cases:
        tc_id,tc_type = case_id(tc)

        if (tc_id, name) in done:
            continue
        try:
            m.measure(tc)
            rec = {"test_case_id": tc_id, "test_case_type": tc_type, "metric_name": name,
                   "score": m.score, "reason": m.reason,
                   "success": m.is_successful(), "error": None}
            print(f"✓ {tc_id} {name} = {m.score}")
        except Exception as e:
            rec = {"test_case_id": tc_id, "test_case_type": tc_type, "metric_name": name,
                   "score": None, "reason": None,
                   "success": None, "error": repr(e)}
            print(f"✗ {tc_id} {name}: {e}")
        all_results_3.append(rec)
        json.dump(all_results_3, open(OUT, "w"), indent=2)   # save every measurement

In [ ]:
def serialize_result(result, tc_id):
    entry = {"id": tc_id, "metrics": []}
    for test_result in result.test_results:
        for metric_data in test_result.metrics_data:
            entry["metrics"].append({
                "name": metric_data.name,
                "score": metric_data.score,
                "success": metric_data.success,
                "reason": metric_data.reason,
            })
    return entry

In [ ]:
from time import sleep


all_results = []
tc_witherror = []

def evaluate_test_cases(tc_id, tc):
    print(f"Running {tc_id}...")
    try:

        result = evaluate(
            test_cases=[tc],
            metrics= metrics,
            # [
            #     faithfulness, answer_rel, hallucination,
            #     tool_correctness, argument_correctness, task_completion,
            #     contextual_recall, contextual_precision, contextual_relevancy,
            #     misuse,
            # ],
            async_config=AsyncConfig(run_async=False),
        )

        serialized = serialize_result(result, tc_id)
        all_results.append(serialized)


    except Exception as e:
        print(f"  ✗ {tc_id} failed with error: {e}")
        tc_witherror.append(tc_id)
        sleep(65)  # optional: wait a bit before continuing to avoid rapid-fire errors
        return

    print(f"  ✓ {tc_id} done — {len(all_results)}/{len(loaded_cases)} saved so far")



In [ ]:
all_results = []

try:
    for id, tc in loaded_cases:
        evaluate_test_cases(id, tc)
        
    if tc_witherror:
        print(f"\nThe following test cases failed with errors: {tc_witherror}")
        for tc_id, tc in loaded_cases:
            if tc_id in tc_witherror:
                evaluate_test_cases(tc_id, tc)
                print(f"  - {tc_id}: {tc}")
                tc_witherror.remove(tc_id)  # remove from the list after re-evaluation

except Exception as e:
    print(f"An unexpected error occurred: {e}")

finally:
    # save after every case — so a crash on case 8 doesn't lose 1-7
    filename = "all_results.json"
    counter = 1
    while True:
        try:
            filename = f"all_results_{counter}.json"
            with open(filename, "x", encoding="utf-8") as f:
                json.dump(all_results, f, indent=4, default=str)
            print(f"  ✓ {tc_id} results saved to {filename}")
            break
        except FileExistsError:
            counter += 1


print("\nAll done. Final file: all_results.json")

In [ ]:
def dict_to_toolcall(d: dict) -> ToolCall:
    return ToolCall(
        name=d["name"],
        input_parameters=d.get("input_parameters") or {},
        output=d.get("output"),
    )

def dict_to_testcase(d: dict) -> LLMTestCase:
    return LLMTestCase(
        input=d["input"],
        actual_output=d["actual_output"],
        expected_output=d.get("expected_output"),
        context=d.get("context"),
        retrieval_context=d.get("retrieval_context"),
        tools_called=[dict_to_toolcall(t) for t in d.get("tools_called", [])],
        expected_tools=[dict_to_toolcall(t) for t in d.get("expected_tools", [])],
    )

with open("test_cases_built.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# keep id + LLMTestCase paired together
loaded_cases = [(d["id"], dict_to_testcase(d)) for d in raw]

print(f"Loaded {len(loaded_cases)} test cases")

In [ ]:
import json, traceback
from pathlib import Path

RESULTS_FILE = Path("all_results_3.json")
all_results = []
failed = []

def save():
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_results_3, f, indent=2, default=str)

def run_one(tc_id, tc):
    evaluate_test_cases(tc_id, tc)   # must append into all_results
    save()

for m, tc in zip(meta, test_cases):
    tc_id = m["id"]
    try:
        run_one(tc_id, tc)
        print(f"✓ {tc_id} done ({len(all_results)} rows)")
    except Exception as e:
        failed.append(tc_id)
        print(f"✗ {tc_id} failed: {type(e).__name__}: {e}")
        with open("failures.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "id": tc_id,
                "error": repr(e),
                "trace": traceback.format_exc(),
            }) + "\n")

# retry pass over failures only
for m, tc in zip(meta, test_cases):
    tc_id = m["id"]
    if tc_id not in failed:
        continue
    try:
        run_one(tc_id, tc)
        failed.remove(tc_id)
        print(f"✓ {tc_id} recovered")
    except Exception as e:
        print(f"✗ {tc_id} failed again: {type(e).__name__}: {e}")

print(f"\nDone. {len(all_results)} rows → {RESULTS_FILE}")
print(f"Still failing: {failed or 'none'}")

In [ ]:
for md in results.test_results[i].metrics_data:
    all_results.append({
        "id": tc_id,
        "type": m["type"],
        "metric": md.name,
        "score": md.score,
        "success": md.success,
        "reason": md.reason,
    })